In [204]:
import pysam, seaborn, re, scipy, os
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from scipy.stats import ranksums
import warnings

# Assuming SmallSampleWarning is the type of warning you want to ignore
warnings.filterwarnings("ignore", category=UserWarning, message=".*SmallSampleWarning.*")

pd.options.display.float_format = '{:.6f}'.format



In [84]:
def get_base_modification_dictionary(
    bam_file, ref_seq, chromosome, phase_region, sv_supporting_reads
):
    """
    This is for the first time phasing within SNP phased block
    Return value: a dictionary that contains cpg location and its haplotype related base modification score
    """
    methylation_identifier_0 = ('C', 0, 'm') # We only care about 5mc!!!
    methylation_identifier_1 = ('C', 1, 'm')
    phase_region_start = phase_region[0]
    phase_region_end = phase_region[1]

    phased_block_ref = ref_seq.fetch(
        chromosome, phase_region_start, phase_region_end)
    cg_loc = [
        m.start(0) for m in re.finditer("CG", str(phased_block_ref))
    ]  # Use regular expression to find all CpG locations on the reference
    # record G locatioon of 'CG's
    cg_loc = [x + phase_region_start + 1 for x in cg_loc]
    hp_myth_dict = dict()
    """
        Data structure:
        {i:[[list1], [list2], num_1, num_2]} 
            i: CpG locations
            list1: ML score for SV-supporting reads
            list2: ML score for non SV-supporting reads
            num1: coverage for SV-supporting reads
            num2: coverage for non SV-supporting reads
        Use dictionary so that when querying CpG locations the time complexity is O(1)
    """
    for i in cg_loc:
        # build the dictionary
        hp_myth_dict.update({i: [[],[],0,0]})
    phased_block_alignment = bam_file.fetch(
        chromosome, phase_region[0], phase_region[1], multiple_iterators=True
    )

    for reads in phased_block_alignment:
        read_base_ref_loc = reads.get_reference_positions(full_length=True)  
        # use full_length=True or the positions won't match
        mm = (reads.modified_bases)
        # mm is a dictionary that contains {score type: [(location, score)]}. score is 255-based
        if (mm != -1) and (mm != {}):  # update base modification scores list
            if methylation_identifier_0 in list(mm.keys()):
                methylation_identifier = methylation_identifier_0
            elif methylation_identifier_1 in list(mm.keys()):
                methylation_identifier = methylation_identifier_1
            for i in mm[methylation_identifier]:  # Remora only output one type of score: c 1 m/c 0 m, but this part can be improved for other methlyation callers
                if read_base_ref_loc[i[0]]:  # i format: (loc, score)
                    if reads.is_forward:  # cg/gc on forward and reverse reads
                        mm_ref_loc = read_base_ref_loc[i[0]] + 1
                    else:
                        mm_ref_loc = read_base_ref_loc[i[0]]
                    if mm_ref_loc in hp_myth_dict.keys():                       
                        modification_chance = i[1]  # 0 - 255 base d
                        if reads.query_name in sv_supporting_reads:
                            hp_myth_dict[mm_ref_loc][0].append(modification_chance)
                            hp_myth_dict[mm_ref_loc][2] += 1 
                        else:
                            hp_myth_dict[mm_ref_loc][1].append(modification_chance)
                            hp_myth_dict[mm_ref_loc][3] += 1 
        # for i in read_base_ref_loc:
        #     if i in hp_myth_dict.keys():  # O(1) search
        #         hp_myth_dict[i][2] += 1 
    return hp_myth_dict

In [212]:
def get_base_modification_dictionary_basic(
    bam_file, ref_seq, chromosome, phase_region
):
    """
    This is for the first time phasing within SNP phased block
    Return value: a dictionary that contains cpg location and its haplotype related base modification score
    """
    methylation_identifier_0 = ('C', 0, 'm') # We only care about 5mc!!!
    methylation_identifier_1 = ('C', 1, 'm')
    phase_region_start = phase_region[0]
    phase_region_end = phase_region[1]

    phased_block_ref = ref_seq.fetch(
        chromosome, phase_region_start, phase_region_end)
    cg_loc = [
        m.start(0) for m in re.finditer("CG", str(phased_block_ref))
    ]  # Use regular expression to find all CpG locations on the reference
    # record G locatioon of 'CG's
    cg_loc = [x + phase_region_start + 1 for x in cg_loc]
    hp_myth_dict = dict()
    """
        Data structure:
        {i:[[], num_1]} 
            i: CpG locations
            list1: ML score for SV-supporting reads
            num1: coverage for SV-supporting reads
        Use dictionary so that when querying CpG locations the time complexity is O(1)
    """
    for i in cg_loc:
        # build the dictionary
        hp_myth_dict.update({i: [[],0]})
    phased_block_alignment = bam_file.fetch(
        chromosome, phase_region[0], phase_region[1], multiple_iterators=True
    )

    for reads in phased_block_alignment:
        if not reads.is_secondary and not reads.is_supplementary:
            read_base_ref_loc = reads.get_reference_positions(full_length=True)  
            # use full_length=True or the positions won't match
            mm = (reads.modified_bases)
            # mm is a dictionary that contains {score type: [(location, score)]}. score is 255-based
            if (mm != -1) and (mm != {}):  # update base modification scores list
                if methylation_identifier_0 in list(mm.keys()):
                    methylation_identifier = methylation_identifier_0
                elif methylation_identifier_1 in list(mm.keys()):
                    methylation_identifier = methylation_identifier_1
                for i in mm[methylation_identifier]:  # Remora only output one type of score: c 1 m/c 0 m, but this part can be improved for other methlyation callers
                    if read_base_ref_loc[i[0]]:  # i format: (loc, score)
                        if reads.is_forward:  # cg/gc on forward and reverse reads
                            mm_ref_loc = read_base_ref_loc[i[0]] + 1
                        else:
                            mm_ref_loc = read_base_ref_loc[i[0]]
                        if mm_ref_loc in hp_myth_dict.keys():                       
                            modification_chance = i[1]  # 0 - 255 base d
                            hp_myth_dict[mm_ref_loc][0].append(modification_chance)
                            hp_myth_dict[mm_ref_loc][1] += 1 

    return hp_myth_dict

In [156]:
def read_vcf_to_df(vcf_file):
    records = []
    
    for record in vcf_file.fetch():
        if record.info["SVTYPE"] in ["INS", "DEL"]:
            df_record = {
                "chr": record.chrom,
                "location": record.pos,
                "id": record.id,
                "sv_len": record.info.get("SVLEN", "NA"),
                "supporting_reads": record.info.get("RNAMES", "NA"),
                "stdev_len": record.info.get("STDEV_LEN", "NA"),
                "stdev_pos": record.info.get("STDEV_POS", "NA"),
            }

            stdev_pos = df_record["stdev_pos"]
            sv_len = df_record["sv_len"] if record.info["SVTYPE"] == "DEL" else 0
            stdev_len = df_record["stdev_len"] 
            ref_start = df_record["location"] - stdev_pos if stdev_pos != "NA" else np.nan
            ref_end = df_record["location"] + stdev_pos + stdev_len - sv_len if stdev_pos != "NA" else np.nan

            df_record.update({"ref_start": int(ref_start), "ref_end": int(ref_end)})
            records.append(df_record)
    
    sv_df = pd.DataFrame(records, columns=["chr", "location", "id", "sv_len", "supporting_reads", "stdev_len", "stdev_pos", "ref_start", "ref_end"])

    return sv_df


In [160]:
def calculate_ranksum(data_dict):
    result = {}
    
    for i, values in data_dict.items():
        list1 = values[0]
        list2 = values[1]
        
        # Perform ranksum test but remove the warning 
        if len(list1) >= 2 and len(list2) >= 2:
            _, p_value = ranksums(list1, list2)
        else:
            p_value = np.nan 
        
        # Store the p-value in the result dictionary
        result[i] = p_value
    
    return result


In [217]:
def calculate_ranksum_hprc(data_dict):
    result = {}
    
    for i, values in data_dict.items():
        list1 = values[0]
        list2 = values[2]
        
        # Perform ranksum test but remove the warning 
        if len(list1) >= 2 and len(list2) >= 2:
            _, p_value = ranksums(list1, list2)
        else:
            p_value = np.nan 
        
        # Store the p-value in the result dictionary
        result[i] = p_value
    
    return result


In [184]:
def calculate_methylation_diff_region(sv_vcf, input_bam, reference_genome, min_supporting_read_num = 5, sv_discovery_range=1000):
    hapmap_ref_file = pysam.AlignmentFile(input_bam)
    referece_sequence = pysam.Fastafile(reference_genome)
    filtered_mosaic_sv = pysam.VariantFile(sv_vcf)
    filtered_mosaic_sv_df = read_vcf_to_df(filtered_mosaic_sv)

    cpg_diff_percentages = []
    
    filtered_mosaic_sv_df_filtered = filtered_mosaic_sv_df[filtered_mosaic_sv_df.supporting_reads.apply(len)>min_supporting_read_num]
    # filtered_mosaic_sv_df_filtered = filtered_mosaic_sv_df_filtered.head(10) # debug 
    for _, test_sv in tqdm(filtered_mosaic_sv_df_filtered.iterrows(), total=len(filtered_mosaic_sv_df_filtered), desc="Processing SVs"):
        modification_dict = get_base_modification_dictionary(bam_file=hapmap_ref_file, 
                                                             ref_seq=referece_sequence, 
                                                             chromosome=test_sv.chr, 
                                                             phase_region=(int(test_sv.ref_start)-sv_range, int(test_sv.ref_end)+sv_discovery_range), 
                                                             sv_supporting_reads=test_sv.supporting_reads)
        ranksum_dict = calculate_ranksum(modification_dict) # nan if list length is 0
        cpg_diff_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value < 0.05}
        cpg_same_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value > 0.05}
        # Calculate the percentage of different CpG locations
        if len(cpg_diff_locs) + len(cpg_same_locs) > 0:
            cpg_diff_percentage = len(cpg_diff_locs) / (len(cpg_diff_locs) + len(cpg_same_locs))
            # print(cpg_diff_percentage)
        else:
            cpg_diff_percentage = None  # or some default value
        
        cpg_diff_percentages.append(cpg_diff_percentage)    
    # print(cpg_diff_percentages)
    filtered_mosaic_sv_df_filtered['cpg_diff_percentage'] = cpg_diff_percentages

    return filtered_mosaic_sv_df_filtered


In [185]:
tp_comp_sv = "hapmap_sv_filtering/sv_benchmarking_collapsed_filtered_hetero_mosiac_bench_new/tp-comp.vcf.gz"
hapmap_bam = "hapmap_ont_primary.bam"
ref_genome = "../../references/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa"
cpg_diff_df = calculate_methylation_diff_region(sv_vcf=tp_comp_sv, input_bam=hapmap_bam,reference_genome=ref_genome, min_supporting_read_num=0)

Processing SVs: 100%|██████████| 5001/5001 [17:31<00:00,  4.76it/s]


In [186]:
cpg_diff_df

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage
0,chr1,711979,Sniffles2.DEL.2195S0,-90,"(27c14a23-3b4d-49f6-a7dd-0fd28369fac5, f849656...",0.000000,0.000000,711979,712069,0.033333
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658,0.276471
2,chr1,934055,Sniffles2.DEL.21BBS0,-307,"(9ce33956-9322-42b4-ba60-14c5916c340c, 13e33a1...",11.971000,27.253000,934027,934401,0.302752
3,chr1,1076228,Sniffles2.DEL.21D5S0,-538,"(bc20e096-4d3c-4f83-9335-46084db73a87, 7403017...",21.639999,30.363001,1076197,1076818,0.323232
4,chr1,1108690,Sniffles2.DEL.21E1S0,-280,"(2428e969-ba42-41b2-b25e-3d7af6f9b446, 5f284ee...",0.000000,0.000000,1108690,1108970,0.037736
...,...,...,...,...,...,...,...,...,...,...
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638,0.125000
4997,chr22,50521320,Sniffles2.DEL.191FS15,-65,"(201690d4-4bc5-4ebc-878e-5287f2b1629a, 51492bd...",0.000000,0.000000,50521320,50521385,0.041667
4998,chr22,50608338,Sniffles2.INS.8FAS15,57,"(2a1abb42-5c28-4f8f-84e5-bdf220826d43, f2aa744...",0.447000,0.000000,50608338,50608338,0.177419
4999,chr22,50685534,Sniffles2.DEL.1934S15,-148,"(5c43ddbd-092f-47ce-83af-7e32590a5f8d, 01a3448...",0.447000,20.354000,50685513,50685702,0.054545


In [168]:
filtered_mosaic_sv = pysam.VariantFile("hapmap_sv_filtering/sv_benchmarking_collapsed_filtered_hetero_mosiac_bench_new/tp-comp.vcf.gz")

In [169]:
filtered_mosaic_sv_df = read_vcf_to_df(filtered_mosaic_sv)

In [173]:
# filtered_mosaic_sv_df_filtered = filtered_mosaic_sv_df[filtered_mosaic_sv_df.supporting_reads.apply(len)>5]
# filtered_mosaic_sv_df_filtered

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658
2,chr1,934055,Sniffles2.DEL.21BBS0,-307,"(9ce33956-9322-42b4-ba60-14c5916c340c, 13e33a1...",11.971000,27.253000,934027,934401
3,chr1,1076228,Sniffles2.DEL.21D5S0,-538,"(bc20e096-4d3c-4f83-9335-46084db73a87, 7403017...",21.639999,30.363001,1076197,1076818
6,chr1,1184758,Sniffles2.INS.7DS0,753,"(9d195e7c-a01e-45a0-a1e1-36de3e981f07, 4fbde2d...",3.317000,45.229000,1184712,1184806
7,chr1,1479431,Sniffles2.DEL.221FS0,-52,"(d61f6850-4ddc-4cd2-bb7a-227c0ec03559, ad3a851...",0.000000,0.000000,1479431,1479483
...,...,...,...,...,...,...,...,...,...
4989,chr22,50009895,Sniffles2.INS.8B2S15,759,"(99bfa973-5d40-4ae6-bc25-deb160836785, c994022...",1.453000,2.646000,50009892,50009899
4992,chr22,50081034,Sniffles2.INS.8BDS15,50,"(291e487b-5766-46d7-95fa-db1d350dd2b0, 421e732...",0.408000,1.633000,50081032,50081036
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638
4998,chr22,50608338,Sniffles2.INS.8FAS15,57,"(2a1abb42-5c28-4f8f-84e5-bdf220826d43, f2aa744...",0.447000,0.000000,50608338,50608338


In [74]:
hapmap_ref_file = pysam.AlignmentFile("hapmap_ont_primary.bam")
referece_sequence = pysam.Fastafile("../../references/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa")

In [178]:
sv_range = 1000
test_sv = filtered_mosaic_sv_df.iloc[7]
test_sv

chr                                                              chr1
location                                                      1479431
id                                               Sniffles2.DEL.221FS0
sv_len                                                            -52
supporting_reads    (d61f6850-4ddc-4cd2-bb7a-227c0ec03559, ad3a851...
stdev_len                                                    0.000000
stdev_pos                                                    0.000000
ref_start                                                     1479431
ref_end                                                       1479483
hprc_loc                                                      1479430
hprc_len                                                          -52
hprc_id                                          HapMapBench.00000261
Name: 7, dtype: object

In [179]:
modification_dict = get_base_modification_dictionary(bam_file=hapmap_ref_file, ref_seq=referece_sequence, chromosome=test_sv.chr, phase_region=(int(test_sv.ref_start)-sv_range, int(test_sv.ref_end)+sv_range), sv_supporting_reads=test_sv.supporting_reads)

In [180]:
int(test_sv.ref_start), int(test_sv.ref_end)

(1479431, 1479483)

In [183]:
ranksum_dict = calculate_ranksum(modification_dict)
cpg_diff_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value < 0.05}
cpg_same_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value > 0.05}
# ranksum_dict, 
len(cpg_diff_locs), len(cpg_same_locs), len(cpg_diff_locs) / (len(cpg_diff_locs) + len(cpg_same_locs))

(3, 58, 0.04918032786885246)

In [199]:
def read_vcf_to_df_comp(vcf_file):
    records = []
    sample_list = ["HG002", "HG00438", "HG005", "HG02257", "HG02486", "HG02622"]
    for record in vcf_file.fetch():
        if record.info["SVTYPE"] in ["INS", "DEL"]:
            df_record = {
                "chr": record.chrom,
                "location": record.pos,
                "id": record.id,
                "sv_len": record.info.get("SVLEN", "NA"),}
            for sample_name in sample_list:
                if record.samples[sample_name]['GT'] != (0,0):
                # df_record.update({sample_name:record.samples[sample_name]['GT']})
                    df_record.update({'sample_name' : sample_name})
            # print(df_record)
            records.append(df_record)

    # sv_df = pd.DataFrame(records, columns=["chr", "location", "id", "sv_len", "supporting_reads", "stdev_len", "stdev_pos", "ref_start", "ref_end"])
    # sv_df = pd.DataFrame(records, columns=["chr", "location", "id", "sv_len"]+ sample_list)
    sv_df = pd.DataFrame(records, columns=["chr", "location", "id", "sv_len", "sample_name"])
    return sv_df


In [200]:
hprc_tp_sv = pysam.VariantFile("hapmap_sv_filtering/sv_benchmarking_collapsed_filtered_hetero_mosiac_bench_new/tp-base.vcf.gz")
hprc_tp_sv_df = read_vcf_to_df_comp(hprc_tp_sv)
hprc_tp_sv_df

,chr,location,id,sv_len,sample_name
0,chr1,711958,HapMapBench.00000006,-90,HG02622
1,chr1,904656,HapMapBench.00000031,354,HG02622
2,chr1,933961,HapMapBench.00000037,-280,HG02622
3,chr1,1076227,HapMapBench.00000089,-538,HG02622
4,chr1,1108628,HapMapBench.00000129,-280,HG02622
...,...,...,...,...,...
4996,chr22,50405556,HapMapBench.00106422,-59,HG005
4997,chr22,50521319,HapMapBench.00106454,-65,HG02257
4998,chr22,50608337,HapMapBench.00106458,57,HG02622
4999,chr22,50685533,HapMapBench.00106492,-148,HG02622


In [201]:
cpg_diff_df[['hprc_loc', 'hprc_len', 'hprc_id', 'hprc_sample_name']] = hprc_tp_sv_df[['location', 'sv_len', 'id', 'sample_name']]
cpg_diff_df

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name
0,chr1,711979,Sniffles2.DEL.2195S0,-90,"(27c14a23-3b4d-49f6-a7dd-0fd28369fac5, f849656...",0.000000,0.000000,711979,712069,0.033333,711958,-90,HapMapBench.00000006,HG02622
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658,0.276471,904656,354,HapMapBench.00000031,HG02622
2,chr1,934055,Sniffles2.DEL.21BBS0,-307,"(9ce33956-9322-42b4-ba60-14c5916c340c, 13e33a1...",11.971000,27.253000,934027,934401,0.302752,933961,-280,HapMapBench.00000037,HG02622
3,chr1,1076228,Sniffles2.DEL.21D5S0,-538,"(bc20e096-4d3c-4f83-9335-46084db73a87, 7403017...",21.639999,30.363001,1076197,1076818,0.323232,1076227,-538,HapMapBench.00000089,HG02622
4,chr1,1108690,Sniffles2.DEL.21E1S0,-280,"(2428e969-ba42-41b2-b25e-3d7af6f9b446, 5f284ee...",0.000000,0.000000,1108690,1108970,0.037736,1108628,-280,HapMapBench.00000129,HG02622
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638,0.125000,50405556,-59,HapMapBench.00106422,HG005
4997,chr22,50521320,Sniffles2.DEL.191FS15,-65,"(201690d4-4bc5-4ebc-878e-5287f2b1629a, 51492bd...",0.000000,0.000000,50521320,50521385,0.041667,50521319,-65,HapMapBench.00106454,HG02257
4998,chr22,50608338,Sniffles2.INS.8FAS15,57,"(2a1abb42-5c28-4f8f-84e5-bdf220826d43, f2aa744...",0.447000,0.000000,50608338,50608338,0.177419,50608337,57,HapMapBench.00106458,HG02622
4999,chr22,50685534,Sniffles2.DEL.1934S15,-148,"(5c43ddbd-092f-47ce-83af-7e32590a5f8d, 01a3448...",0.447000,20.354000,50685513,50685702,0.054545,50685533,-148,HapMapBench.00106492,HG02622


In [254]:
hapmap_ref_file = pysam.AlignmentFile("hapmap_ont_primary.bam")
referece_sequence = pysam.Fastafile("../../references/GCA_000001405.15_GRCh38_no_alt_analysis_set.fa")
sv_discovery_range = 1000
cpg_diff_percentages = []

cpg_diff_df_test = cpg_diff_df # debug 
for _, test_sv in tqdm(cpg_diff_df_test.iterrows(), total=len(cpg_diff_df_test), desc="Processing SVs"):
    # test_sv = cpg_diff_df_test.iloc[6]
    hprc_bam_loc = os.path.join("hprc_data/", f"{test_sv.hprc_sample_name}.sorted.bam")
    hprc_bam = pysam.AlignmentFile(hprc_bam_loc)
    modification_dict_hprc = get_base_modification_dictionary_basic(bam_file=hprc_bam, 
                                                         ref_seq=referece_sequence, 
                                                         chromosome=test_sv.chr, 
                                                         phase_region=(int(test_sv.ref_start)-sv_discovery_range, int(test_sv.ref_end)+sv_discovery_range), )
    modification_dict_hapmap = get_base_modification_dictionary_basic(bam_file=hapmap_ref_file, 
                                                         ref_seq=referece_sequence, 
                                                         chromosome=test_sv.chr, 
                                                         phase_region=(int(test_sv.ref_start)-sv_discovery_range, int(test_sv.ref_end)+sv_discovery_range), )
    ranksum_dict = {}
    for key in modification_dict_hprc.keys():
        hprc_ml_list = modification_dict_hprc[key][0]
        hapmap_ml_list = modification_dict_hapmap[key][0]
        _, p_value = ranksums(hprc_ml_list, hapmap_ml_list)
        ranksum_dict.update({key: p_value})
    cpg_diff_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value < 0.05}
    cpg_same_locs = {i: p_value for i, p_value in ranksum_dict.items() if p_value > 0.05}
    # #     # Calculate the percentage of different CpG locations
    if len(cpg_diff_locs) + len(cpg_same_locs) > 0:
        cpg_diff_percentage = len(cpg_diff_locs) / (len(cpg_diff_locs) + len(cpg_same_locs))
    else:
        cpg_diff_percentage = None  # or some default value

    cpg_diff_percentages.append(cpg_diff_percentage)
# print(cpg_diff_percentages)
cpg_diff_df_test['cpg_diff_percentage_hprc'] = cpg_diff_percentages

# test_sv

Processing SVs:   0%|          | 4/5001 [00:02<57:32,  1.45it/s]  /space1/tmp/slurm.674210/ipykernel_401155/4039126352.py:23: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  _, p_value = ranksums(hprc_ml_list, hapmap_ml_list)
Processing SVs: 100%|██████████| 5001/5001 [57:57<00:00,  1.44it/s] 


In [290]:
cpg_diff_df_test_filtered = cpg_diff_df_test[cpg_diff_df_test.supporting_reads.apply(len) > 5]
# cpg_diff_df_test_filtered[cpg_diff_df_test_filtered.cpg_diff_percentage > 0.1]
# cpg_diff_df_test_filtered[cpg_diff_df_test_filtered.cpg_diff_percentage_hprc > 0.5]
cpg_diff_df_test_filtered[(cpg_diff_df_test_filtered.cpg_diff_percentage > 0.1) & (cpg_diff_df_test_filtered.cpg_diff_percentage_hprc > 0.5)]

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name,cpg_diff_percentage_hprc
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658,0.276471,904656,354,HapMapBench.00000031,HG02622,0.882353
6,chr1,1184758,Sniffles2.INS.7DS0,753,"(9d195e7c-a01e-45a0-a1e1-36de3e981f07, 4fbde2d...",3.317000,45.229000,1184712,1184806,0.089744,1184655,756,HapMapBench.00000152,HG02622,0.525641
9,chr1,1508184,Sniffles2.DEL.2223S0,-2201,"(0ac10069-db58-4d23-9873-385c683fe1bd, 20fe089...",0.000000,0.000000,1508184,1510385,0.127660,1508183,-2201,HapMapBench.00000266,HG02622,0.671756
10,chr1,1605517,Sniffles2.INS.A8S0,275,"(b8abd015-a24b-4572-962e-f02cfdcc6179, 53844ba...",8.848000,101.568001,1605415,1605627,0.100000,1605516,275,HapMapBench.00000287,HG02486,0.704918
12,chr1,2094385,Sniffles2.DEL.2264S0,-245,"(8290ee91-1347-44ba-b660-605debaa4be2, cea7fde...",0.447000,4.472000,2094380,2094634,0.093023,2094384,-245,HapMapBench.00000333,HG02622,0.525862
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4986,chr22,49922747,Sniffles2.INS.8A2S15,144,"(e9847570-6ba2-4922-8d10-ce59337eedb3, b75b46a...",0.983000,2.858000,49922744,49922750,0.072165,49922746,145,HapMapBench.00106264,HG02622,0.693878
4988,chr22,49951170,Sniffles2.INS.8A9S15,187,"(a1267735-77f6-49e8-8a9f-af26d116ec44, d6d1eaa...",0.837000,18.982000,49951151,49951189,0.228070,49951169,185,HapMapBench.00106280,HG02622,0.551724
4989,chr22,50009895,Sniffles2.INS.8B2S15,759,"(99bfa973-5d40-4ae6-bc25-deb160836785, c994022...",1.453000,2.646000,50009892,50009899,0.130000,50009894,756,HapMapBench.00106293,HG02622,0.584158
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638,0.125000,50405556,-59,HapMapBench.00106422,HG005,0.700000


In [306]:
cpg_diff_df_test_filtered[cpg_diff_df_test_filtered.cpg_diff_percentage > 0.1]

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name,cpg_diff_percentage_hprc
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658,0.276471,904656,354,HapMapBench.00000031,HG02622,0.882353
2,chr1,934055,Sniffles2.DEL.21BBS0,-307,"(9ce33956-9322-42b4-ba60-14c5916c340c, 13e33a1...",11.971000,27.253000,934027,934401,0.302752,933961,-280,HapMapBench.00000037,HG02622,0.411765
3,chr1,1076228,Sniffles2.DEL.21D5S0,-538,"(bc20e096-4d3c-4f83-9335-46084db73a87, 7403017...",21.639999,30.363001,1076197,1076818,0.323232,1076227,-538,HapMapBench.00000089,HG02622,0.492647
9,chr1,1508184,Sniffles2.DEL.2223S0,-2201,"(0ac10069-db58-4d23-9873-385c683fe1bd, 20fe089...",0.000000,0.000000,1508184,1510385,0.127660,1508183,-2201,HapMapBench.00000266,HG02622,0.671756
21,chr1,3297895,Sniffles2.INS.188S0,71,"(89d0791f-0ac3-4f7a-bba3-59fb343d38a9, f9d4d55...",0.000000,0.000000,3297895,3297895,0.188679,3297894,71,HapMapBench.00000583,HG02622,0.462963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4979,chr22,49306151,Sniffles2.INS.873S15,149,"(542ed25a-34e1-4173-8778-c22d19f61d09, d0cb967...",5.188000,0.000000,49306151,49306156,0.101695,49306150,160,HapMapBench.00106176,HG02622,0.610169
4988,chr22,49951170,Sniffles2.INS.8A9S15,187,"(a1267735-77f6-49e8-8a9f-af26d116ec44, d6d1eaa...",0.837000,18.982000,49951151,49951189,0.228070,49951169,185,HapMapBench.00106280,HG02622,0.551724
4989,chr22,50009895,Sniffles2.INS.8B2S15,759,"(99bfa973-5d40-4ae6-bc25-deb160836785, c994022...",1.453000,2.646000,50009892,50009899,0.130000,50009894,756,HapMapBench.00106293,HG02622,0.584158
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638,0.125000,50405556,-59,HapMapBench.00106422,HG005,0.700000


In [291]:
cpg_diff_df_test_filtered[cpg_diff_df_test_filtered.cpg_diff_percentage_hprc > 0.5]

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name,cpg_diff_percentage_hprc
1,chr1,904657,Sniffles2.INS.52S0,355,"(fbce6b2f-67ca-4c2f-a66f-0d6e1ae2930e, 54d1278...",1.517000,0.000000,904657,904658,0.276471,904656,354,HapMapBench.00000031,HG02622,0.882353
6,chr1,1184758,Sniffles2.INS.7DS0,753,"(9d195e7c-a01e-45a0-a1e1-36de3e981f07, 4fbde2d...",3.317000,45.229000,1184712,1184806,0.089744,1184655,756,HapMapBench.00000152,HG02622,0.525641
7,chr1,1479431,Sniffles2.DEL.221FS0,-52,"(d61f6850-4ddc-4cd2-bb7a-227c0ec03559, ad3a851...",0.000000,0.000000,1479431,1479483,0.049180,1479430,-52,HapMapBench.00000261,HG02622,0.680000
9,chr1,1508184,Sniffles2.DEL.2223S0,-2201,"(0ac10069-db58-4d23-9873-385c683fe1bd, 20fe089...",0.000000,0.000000,1508184,1510385,0.127660,1508183,-2201,HapMapBench.00000266,HG02622,0.671756
10,chr1,1605517,Sniffles2.INS.A8S0,275,"(b8abd015-a24b-4572-962e-f02cfdcc6179, 53844ba...",8.848000,101.568001,1605415,1605627,0.100000,1605516,275,HapMapBench.00000287,HG02486,0.704918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4988,chr22,49951170,Sniffles2.INS.8A9S15,187,"(a1267735-77f6-49e8-8a9f-af26d116ec44, d6d1eaa...",0.837000,18.982000,49951151,49951189,0.228070,49951169,185,HapMapBench.00106280,HG02622,0.551724
4989,chr22,50009895,Sniffles2.INS.8B2S15,759,"(99bfa973-5d40-4ae6-bc25-deb160836785, c994022...",1.453000,2.646000,50009892,50009899,0.130000,50009894,756,HapMapBench.00106293,HG02622,0.584158
4992,chr22,50081034,Sniffles2.INS.8BDS15,50,"(291e487b-5766-46d7-95fa-db1d350dd2b0, 421e732...",0.408000,1.633000,50081032,50081036,0.021739,50081033,50,HapMapBench.00106329,HG02622,0.531915
4996,chr22,50405558,Sniffles2.DEL.1913S15,-58,"(87ffb65c-a847-497b-851d-f6c366742952, 590fe90...",0.516000,21.566999,50405536,50405638,0.125000,50405556,-59,HapMapBench.00106422,HG005,0.700000


In [315]:
cpg_diff_df_test_filtered[(cpg_diff_df_test_filtered.cpg_diff_percentage > 0.1) & (cpg_diff_df_test_filtered.cpg_diff_percentage_hprc < 0.5)]

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name,cpg_diff_percentage_hprc
2,chr1,934055,Sniffles2.DEL.21BBS0,-307,"(9ce33956-9322-42b4-ba60-14c5916c340c, 13e33a1...",11.971000,27.253000,934027,934401,0.302752,933961,-280,HapMapBench.00000037,HG02622,0.411765
3,chr1,1076228,Sniffles2.DEL.21D5S0,-538,"(bc20e096-4d3c-4f83-9335-46084db73a87, 7403017...",21.639999,30.363001,1076197,1076818,0.323232,1076227,-538,HapMapBench.00000089,HG02622,0.492647
21,chr1,3297895,Sniffles2.INS.188S0,71,"(89d0791f-0ac3-4f7a-bba3-59fb343d38a9, f9d4d55...",0.000000,0.000000,3297895,3297895,0.188679,3297894,71,HapMapBench.00000583,HG02622,0.462963
47,chr1,10891988,Sniffles2.INS.311S0,71,"(439c55ed-f4ac-4242-80ec-3a94c7994128, d9fedef...",2.074000,0.000000,10891988,10891990,0.117647,10891987,75,HapMapBench.00001195,HG02622,0.400000
58,chr1,14793649,Sniffles2.DEL.26ACS0,-87,"(e25cb05e-72a6-4d68-9bea-2a25e277cb7d, 22a59d1...",0.000000,2.041000,14793646,14793738,0.114286,14793648,-87,HapMapBench.00001339,HG02622,0.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4929,chr22,34241300,Sniffles2.INS.5D4S15,317,"(06b5fbc8-640a-40bf-a831-48dca325c4c7, c130460...",4.082000,0.000000,34241300,34241304,0.166667,34241299,325,HapMapBench.00104915,HG02622,0.083333
4958,chr22,44884023,Sniffles2.INS.757S15,186,"(9fb1767a-7594-46f1-8f47-006126739cf9, 7b1ac42...",0.000000,0.000000,44884023,44884023,0.187500,44884022,186,HapMapBench.00105518,HG02622,0.382353
4959,chr22,45568705,Sniffles2.INS.77AS15,435,"(1e4363ee-8464-45f4-8600-9178f72d1d7c, dbc56ab...",8.042000,27.500000,45568677,45568740,0.157895,45568490,404,HapMapBench.00105585,HG02622,0.473684
4969,chr22,48322399,Sniffles2.INS.816S15,64,"(0545f606-fbd3-44d9-8c16-d0b457666fef, 3476cb3...",0.000000,16.500000,48322382,48322415,0.117647,48322398,64,HapMapBench.00105910,HG02622,0.450000


In [308]:
cpg_diff_df_test_filtered[(cpg_diff_df_test_filtered.cpg_diff_percentage < 0.1) & (cpg_diff_df_test_filtered.cpg_diff_percentage_hprc > 0.5)]

,chr,location,id,sv_len,supporting_reads,stdev_len,stdev_pos,ref_start,ref_end,cpg_diff_percentage,hprc_loc,hprc_len,hprc_id,hprc_sample_name,cpg_diff_percentage_hprc
6,chr1,1184758,Sniffles2.INS.7DS0,753,"(9d195e7c-a01e-45a0-a1e1-36de3e981f07, 4fbde2d...",3.317000,45.229000,1184712,1184806,0.089744,1184655,756,HapMapBench.00000152,HG02622,0.525641
7,chr1,1479431,Sniffles2.DEL.221FS0,-52,"(d61f6850-4ddc-4cd2-bb7a-227c0ec03559, ad3a851...",0.000000,0.000000,1479431,1479483,0.049180,1479430,-52,HapMapBench.00000261,HG02622,0.680000
12,chr1,2094385,Sniffles2.DEL.2264S0,-245,"(8290ee91-1347-44ba-b660-605debaa4be2, cea7fde...",0.447000,4.472000,2094380,2094634,0.093023,2094384,-245,HapMapBench.00000333,HG02622,0.525862
19,chr1,3264393,Sniffles2.DEL.231AS0,-51,"(83b03036-a32e-4044-89ae-fdaa895e87f5, 525b226...",1.304000,12.498000,3264380,3264457,0.093023,3264385,-52,HapMapBench.00000578,HG02622,0.558140
23,chr1,3475167,Sniffles2.DEL.2337S0,-240,"(1b69d878-6af0-4105-bc22-6d7579a6d724, c409996...",0.577000,46.828999,3475120,3475454,0.095238,3475101,-239,HapMapBench.00000613,HG02622,0.574468
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4963,chr22,47043976,Sniffles2.INS.7C8S15,126,"(82a70431-f466-4940-a404-fc4f6fdcdc45, ecbf87c...",1.291000,0.000000,47043976,47043977,0.095238,47043975,128,HapMapBench.00105764,HG02622,0.767442
4978,chr22,49219927,Sniffles2.DEL.1877S15,-54,"(7bcbbab7-7793-4357-9d6b-84239e79a84e, 1ff9807...",2.191000,4.243000,49219922,49219987,0.062500,49219926,-54,HapMapBench.00106135,HG02622,0.764706
4985,chr22,49917987,Sniffles2.INS.8A1S15,171,"(a1267735-77f6-49e8-8a9f-af26d116ec44, 0718a12...",0.837000,0.000000,49917987,49917987,0.074074,49917986,190,HapMapBench.00106261,HG00438,0.824818
4986,chr22,49922747,Sniffles2.INS.8A2S15,144,"(e9847570-6ba2-4922-8d10-ce59337eedb3, b75b46a...",0.983000,2.858000,49922744,49922750,0.072165,49922746,145,HapMapBench.00106264,HG02622,0.693878


In [305]:
from scipy.stats import fisher_exact

# Contingency table: [methylated, unmethylated] in group A and B
table = [[100, 20], [1, 4]]

# Perform Fisher's exact test
oddsratio, p_value_fisher = fisher_exact(table)
print(f'Fisher p-value: {p_value_fisher}')

Fisher p-value: 0.0047572759752404015
